# Debug-Run-Vergleich

Laedt alle archivierten Laeufe aus `data/debug_runs/<run_id>/` fuer eine Artikel-URL (oder Domain) und stellt sie als Vergleichstabelle dar — der manuelle Prozess, der diese Session mehrfach per Hand nachgebaut wurde (Qwen-4er-Vergleich, Gemini-3er-Vergleich), jetzt als wiederverwendbares Notebook.

Setzt voraus, dass `analyzer.py` die Debug-Run-Archivierung aus [ADR 0010](../docs/concepts/decisions/0010-quote-amplification-grounding-and-debug-run-history.md) aktiv geschrieben hat (jeder `analyze_article()`-Call seit dem legt automatisch einen `data/debug_runs/<timestamp>_<domain>/`-Ordner an).

**Kein LLM-Call, keine laufende ChromaDB noetig** — liest nur bereits gespeicherte JSON-Dateien.

## 1 — Setup

In [ ]:
import json
from pathlib import Path
import pandas as pd

DEBUG_RUNS = Path("../data/debug_runs")
print(f"{len(list(DEBUG_RUNS.glob('*')))} archivierte Laeufe insgesamt")

## 2 — Laeufe filtern

`URL_FILTER` ist ein Teilstring — matcht gegen `source_url` im gespeicherten Ergebnis. Leer lassen fuer alle Laeufe.

In [ ]:
URL_FILTER = "derstandard.at"  # z.B. Domain oder Teil der URL; "" fuer alle

def load_run(run_dir: Path) -> dict | None:
    f = run_dir / "06_final_result.json"
    if not f.exists():
        return None  # Lauf ist fehlgeschlagen (z.B. Quota/Rate-Limit) — kein Endergebnis
    data = json.loads(f.read_text(encoding="utf-8"))
    data["_run_id"] = run_dir.name
    return data

runs = []
failed = 0
for run_dir in sorted(DEBUG_RUNS.glob("*")):
    if not run_dir.is_dir():
        continue
    data = load_run(run_dir)
    if data is None:
        failed += 1
        continue
    if URL_FILTER and URL_FILTER not in data.get("source_url", ""):
        continue
    runs.append(data)

print(f"{len(runs)} passende, vollstaendige Laeufe gefunden ({failed} fehlgeschlagene Laeufe insgesamt uebersprungen)")

## 3 — Vergleichstabelle

In [ ]:
def stroemung_labels(entry) -> list[str]:
    return [s.get("label") if isinstance(s, dict) else s for s in entry.get("politische_stroemung", [])]

rows = []
for r in runs:
    ft = r.get("framing_target", {})
    techs = r.get("detected_techniques", [])
    pwc = r.get("pass1_word_count") or r.get("word_count", 1)
    rows.append({
        "run_id":              r["_run_id"],
        "provider":            r.get("llm_provider"),
        "model":               r.get("llm_model"),
        "orwell_structural":   ft.get("orwell_index_structural"),
        "quote_amplification": ft.get("quote_amplification_index"),
        "orwell_index":        ft.get("orwell_index"),
        "bernays_score":       round(len(techs) / max(pwc, 1) * 1000, 2),
        "technique_count":     len(techs),
        "dk_index":            ft.get("dunning_kruger_index"),
        "stroemung":           ", ".join(stroemung_labels(r)),
    })

df = pd.DataFrame(rows).sort_values("run_id")
pd.set_option("display.max_colwidth", 60)
df

## 4 — Streubreite

Min/Max/Spannweite pro numerischer Spalte — schneller Blick darauf, ob sich die Werte ueber die Laeufe hinweg tatsaechlich stabilisiert haben.

In [ ]:
numeric_cols = ["orwell_structural", "quote_amplification", "orwell_index", "bernays_score", "dk_index"]
spread = df[numeric_cols].agg(["min", "max"]).T
spread["range"] = spread["max"] - spread["min"]
spread

## 5 — Zwei Laeufe im Detail vergleichen

`RUN_A`/`RUN_B` sind Indizes in `df` (siehe Tabelle oben). Zeigt, welche Techniken nur in einem der beiden Laeufe auftauchen.

In [ ]:
RUN_A = 0
RUN_B = len(runs) - 1

a, b = runs[RUN_A], runs[RUN_B]
print(f"A = {a['_run_id']}  ({a.get('llm_provider')}/{a.get('llm_model')})")
print(f"B = {b['_run_id']}  ({b.get('llm_provider')}/{b.get('llm_model')})")

def tech_set(entry):
    return {(t.get("technique"), t.get("quote", "")[:60]) for t in entry.get("detected_techniques", [])}

only_a = tech_set(a) - tech_set(b)
only_b = tech_set(b) - tech_set(a)

print(f"\nNur in A ({len(only_a)}):")
for tech, quote in sorted(only_a):
    print(f"  [{tech}] {quote}...")

print(f"\nNur in B ({len(only_b)}):")
for tech, quote in sorted(only_b):
    print(f"  [{tech}] {quote}...")

print(f"\nStroemung A: {stroemung_labels(a)}")
print(f"Stroemung B: {stroemung_labels(b)}")